# Asset Total Return Swaps in LUSID

| Section | Topic |
|---|---|
| 1 | Instrument creation |
| 2 | Recipe |
| 3 | Portfolio and transactions |
| 4 | Valuation and accrued interest |
| 5 | Instrument events |

## The instrument

A total return swap has two legs pulling in opposite directions: one side carries the return on
whatever's being referenced (the asset leg), and the other pays the cost of financing that
exposure (the funding leg).

    asset leg     the referenced obligation
    funding leg   a FloatingLeg paying an index plus a spread

The asset leg can be almost anything -- a bond, a loan, an equity, an index. This notebook uses a
`Bond`.

## Matching principal and notional

LUSID checks the two legs against each other: the bond's `principal` has to equal the funding
leg's `notional`. The easiest way to satisfy that at any trade size is to set both to 1.0, and let
the position's own quantity carry the actual size instead.

---
## Setup

In [ ]:
import os
import json
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

from datetime import datetime, timezone, timedelta
import pandas as pd

import lusid
import lusid.models as m
from lusid.extensions import (
    SyncApiClientFactory, SecretsFileConfigurationLoader, EnvironmentVariablesConfigurationLoader)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format

SECRETS_PATH = os.getenv("FBN_SECRETS_PATH") or (
    "secrets.json" if os.path.exists("secrets.json") else None)
config_loaders = ([SecretsFileConfigurationLoader(SECRETS_PATH)] if SECRETS_PATH
                   else [EnvironmentVariablesConfigurationLoader()])

factory = SyncApiClientFactory(config_loaders=config_loaders)


def api(cls):
    return factory.build(cls)


instruments_api   = api(lusid.InstrumentsApi)
txn_portfolio_api = api(lusid.TransactionPortfoliosApi)
portfolios_api    = api(lusid.PortfoliosApi)
quotes_api        = api(lusid.QuotesApi)
recipes_api       = api(lusid.ConfigurationRecipeApi)
aggregation_api   = api(lusid.AggregationApi)

meta = api(lusid.ApplicationMetadataApi).get_lusid_versions()
href = meta.links[0].href
print("Domain      :", href[:href.find("/app/")] if "/app/" in href else href)
print("API version :", meta.build_version)

Domain      : https://fbn-tejan.lusid.com
API version : 0.6.16597.0


---
## Configuration

`SimpleStatic` just prices the position off the quoted mark.

In [2]:
def d(year, month, day):
    return datetime(year, month, day, tzinfo=timezone.utc)


def upsert(key, name, client_internal, definition):
    """Upsert one instrument and return its LUID."""
    resp = instruments_api.upsert_instruments(scope=SCOPE, request_body={
        key: m.InstrumentDefinition(
            name=name,
            identifiers={"ClientInternal": m.InstrumentIdValue(value=client_internal)},
            definition=definition)})
    assert not resp.failed, list(resp.failed.values())[0].detail
    return resp.values[key].lusid_instrument_id


def mastered(luid):
    """Reference an instrument that already exists in the master."""
    return m.MasteredInstrument(
        instrument_type="MasteredInstrument",
        identifiers={"Instrument/default/LusidInstrumentId": luid})


def recreate_portfolio(code, display_name, base_currency, created, recipe=None):
    """Create the portfolio, replacing any earlier run so the book starts empty."""
    request = m.CreateTransactionPortfolioRequest(
        display_name=display_name, code=code, base_currency=base_currency,
        created=created, instrument_scopes=[SCOPE],
        instrument_event_configuration=None if recipe is None else
        m.InstrumentEventConfiguration(
            transaction_template_scopes=["default"],
            recipe_id=m.ResourceId(scope=SCOPE, code=recipe)))
    try:
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Created {SCOPE}/{code}")
    except lusid.ApiException as e:
        if "PortfolioWithIdAlreadyExists" not in str(getattr(e, "body", "")):
            raise
        portfolios_api.delete_portfolio(scope=SCOPE, code=code)
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Recreated {SCOPE}/{code}")


def upsert_price(luid, price, effective, currency):
    """One Price/mid quote, keyed on the instrument's LUID."""
    quotes_api.upsert_quotes(scope=SCOPE, request_body={
        f"{luid}-{effective:%Y%m%d}": m.UpsertQuoteRequest(
            quote_id=m.QuoteId(
                quote_series_id=m.QuoteSeriesId(
                    provider="Lusid", instrument_id=luid,
                    instrument_id_type="LusidInstrumentId",
                    quote_type="Price", field="mid"),
                effective_at=effective.isoformat()),
            metric_value=m.MetricValue(value=price, unit=currency))})


def value(portfolio, effective, metrics, currency, group_by=None):
    """Run the recipe over one portfolio and return the result as a DataFrame."""
    request = m.ValuationRequest(
        recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE),
        metrics=[m.AggregateSpec(key=k, op=op) for k, op in metrics],
        group_by=group_by or ["Instrument/default/Name"],
        report_currency=currency,
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=portfolio, portfolio_entity_type="SinglePortfolio")],
        valuation_schedule=m.ValuationSchedule(effective_at=effective.isoformat()))
    return pd.DataFrame(aggregation_api.get_valuation(valuation_request=request).data)


def transactions(portfolio, from_date, as_at):
    """The portfolio's own booked transactions over a date range, as a DataFrame."""
    txns = txn_portfolio_api.get_transactions(
        scope=SCOPE, code=portfolio,
        from_transaction_date=from_date.isoformat(),
        to_transaction_date=as_at.isoformat()).values
    if not txns:
        return pd.DataFrame(columns=["date", "type", "luid", "units", "consideration"])
    return pd.DataFrame([{
        "date": pd.Timestamp(t.transaction_date).strftime("%Y-%m-%d"),
        "type": t.type,
        "luid": t.instrument_uid,
        "units": t.units,
        "consideration": t.total_consideration.amount,
    } for t in txns]).sort_values(["date", "type"]).reset_index(drop=True)


SCOPE     = "AssetTrsDemo"
RECIPE    = "asset-trs-demo-recipe"
PORTFOLIO = "asset-trs-demo-book"

BOND_ID   = "DEMO-NWIND-BOND-01"
DESC      = "Demo Northwind Industries 6.50% 2029 TRS"
CURRENCY  = "USD"
START     = d(2025, 2, 17)
MATURITY  = d(2029, 2, 17)
ASOF      = d(2025, 12, 15)

COUPON     = 0.0650
FREQUENCY  = "6M"
DAY_COUNT  = "ActActIcma"

FINANCING_INDEX      = "SOFRRATE"
FINANCING_FIXING_REF = "USD-SOFR"
FINANCING_SPREAD     = 0.0045       # added on top of the fixing reference
FINANCING_DAY_COUNT  = "Actual360"
ASSET_SIDE   = "Receive"
FUNDING_SIDE = "Pay"

NOTIONAL        = 1.0               # asset principal == funding notional, see the constraint above
INITIAL_PRICE   = 1.0
RESET_FREQUENCY = "2W"              # reset frequency for the asset leg's reset schedule

QUANTITY = 50_000.00
PRICE    = 2.35                     # points, quoted mark
DENOM    = 100

print(f"{DESC}")
print(f"  asset   {ASSET_SIDE:<8} {BOND_ID} at {COUPON:.3%}, {FREQUENCY} {DAY_COUNT}")
print(f"  funding {FUNDING_SIDE:<8} {FINANCING_INDEX} + {FINANCING_SPREAD:.2%} ({FINANCING_FIXING_REF})")
print(f"  {QUANTITY:,.0f} units at {PRICE} points on {ASOF:%Y-%m-%d}")
print(f"  market value = {QUANTITY:,.0f} x {PRICE} / {DENOM} = {QUANTITY * PRICE / DENOM:,.2f} {CURRENCY}")

Demo Northwind Industries 6.50% 2029 TRS
  asset   Receive  DEMO-NWIND-BOND-01 at 6.500%, 6M ActActIcma
  funding Pay      SOFRRATE + 0.45% (USD-SOFR)
  50,000 units at 2.35 points on 2025-12-15
  market value = 50,000 x 2.35 / 100 = 1,175.00 USD


---
# 1. Instrument creation

The asset leg's `Bond` is built with its own coupon and conventions, just like a standalone bond
would be. `principal` gets the same unit value as the funding leg's `notional`.

In [3]:
asset = m.Bond(
    instrument_type="Bond",
    start_date=START,
    maturity_date=MATURITY,
    dom_ccy=CURRENCY,
    principal=NOTIONAL,
    coupon_rate=COUPON,
    identifiers={"Cusip": BOND_ID},
    flow_conventions=m.FlowConventions(
        currency=CURRENCY,
        payment_frequency=FREQUENCY,
        day_count_convention=DAY_COUNT,
        roll_convention=str(START.day),
        payment_calendars=[], reset_calendars=[]))

trs = m.TotalReturnSwap(
    instrument_type="TotalReturnSwap",
    start_date=START,
    maturity_date=MATURITY,
    asset_leg=m.AssetLeg(
        asset=asset,
        pay_receive=ASSET_SIDE,
        initial_price=INITIAL_PRICE,
        reset_schedule=m.ResetSchedule(frequency=RESET_FREQUENCY),
        income_policy="PassThrough"),
    funding_leg=m.FloatingLeg(
        instrument_type="FloatingLeg",
        start_date=START,
        maturity_date=MATURITY,
        notional=NOTIONAL,
        leg_definition=m.LegDefinition(
            rate_or_spread=FINANCING_SPREAD,
            pay_receive=FUNDING_SIDE,
            conventions=m.FlowConventions(
                currency=CURRENCY,
                payment_frequency=RESET_FREQUENCY,
                day_count_convention=FINANCING_DAY_COUNT,
                roll_convention=str(START.day),
                payment_calendars=[], reset_calendars=[]),
            index_convention=m.IndexConvention(
                currency=CURRENCY,
                payment_tenor="1D",
                fixing_reference=FINANCING_FIXING_REF,
                index_name=FINANCING_INDEX,
                publication_day_lag=0,
                day_count_convention=FINANCING_DAY_COUNT),
            reset_convention="InArrears",
            stub_type="ShortBack",
            notional_exchange_type="None")))

TRS_LUID = upsert("trs", DESC, BOND_ID, trs)
print(f"Asset TRS : {TRS_LUID}")

Asset TRS : LUID_00003DFP


---
# 2. Recipe

With `SimpleStatic`, the position is reported at its quoted mark. The legs' conventions still
describe what the instrument is, but under this model, they don't feed into the price itself.

In [4]:
recipes_api.upsert_configuration_recipe(
    upsert_recipe_request=m.UpsertRecipeRequest(
        configuration_recipe=m.ConfigurationRecipe(
            scope=SCOPE, code=RECIPE,
            description="Asset TRS, marked",
            market=m.MarketContext(
                market_rules=[m.MarketDataKeyRule(
                    key="Quote.LusidInstrumentId.*", supplier="Lusid", data_scope=SCOPE,
                    quote_type="Price", field="mid", quote_interval="1Y")],
                options=m.MarketOptions(
                    default_supplier="Lusid",
                    default_instrument_code_type="LusidInstrumentId",
                    default_scope=SCOPE)),
            pricing=m.PricingContext(
                model_rules=[m.VendorModelRule(
                    supplier="Lusid", model_name="SimpleStatic",
                    instrument_type="TotalReturnSwap")],
                options=m.PricingOptions(allow_partially_successful_evaluation=True)))))

print(f"Recipe: {SCOPE}/{RECIPE}")

Recipe: AssetTrsDemo/asset-trs-demo-recipe


---
# 3. Portfolio and transactions

Striking a TRS doesn't involve any principal changing hands, so `totalConsideration` is zero.

In [5]:
recreate_portfolio(PORTFOLIO, "Asset TRS Demo Book", CURRENCY, d(2025, 1, 1), recipe=RECIPE)

txn_portfolio_api.upsert_transactions(
    scope=SCOPE, code=PORTFOLIO,
    transaction_request=[m.TransactionRequest(
        transaction_id="BUY-TRS",
        type="Buy",
        instrument_identifiers={"Instrument/default/LusidInstrumentId": TRS_LUID},
        transaction_date=START.isoformat(),
        settlement_date=START.isoformat(),
        units=QUANTITY,
        transaction_price=m.TransactionPrice(price=0.0, type="Price"),
        total_consideration=m.CurrencyAndAmount(amount=0.0, currency=CURRENCY),
        source="default")])

display(transactions(PORTFOLIO, START, START))

Recreated AssetTrsDemo/asset-trs-demo-book


,date,type,luid,units,consideration
0,2025-02-17,Buy,LUID_00003DFP,"50,000.00",0.00


---
# 4. Valuation and accrued interest

Just one quote needed, at the swap's own price, expressed per unit. `Valuation/CleanPV` is the
swap's full quoted mark. `Valuation/Accrued` is scoped more narrowly: it's the bond leg's own
coupon accrued since the last reset.

In [6]:
upsert_price(TRS_LUID, PRICE / DENOM, ASOF, CURRENCY)

METRICS = [("Instrument/default/Name", "Value"),
           ("Holding/default/Units",   "Sum"),
           ("Valuation/CleanPV",       "Sum"),
           ("Valuation/Accrued",       "Sum")]

result = value(PORTFOLIO, ASOF, METRICS, CURRENCY)
display(result)

pv = result.loc[result["Instrument/default/Name"] == DESC, "Sum(Valuation/CleanPV)"].iloc[0]
print(f"LUSID CleanPV {pv:,.2f}  vs  quoted mark {QUANTITY * PRICE / DENOM:,.2f}")

,Instrument/default/Name,Sum(Holding/default/Units),Sum(Valuation/CleanPV),Sum(Valuation/Accrued)
0,Demo Northwind Industries 6.50% 2029 TRS,"50,000.00","1,175.00","1,059.78"


LUSID CleanPV 1,175.00  vs  quoted mark 1,175.00


---
# 5. Instrument events

A `TotalReturnSwap` comes with a built-in `MaturityEvent` -- the swap's own `maturity_date` is all
it needs, with nothing extra to configure.

Whether LUSID actually reports that event depends on a separate setting though:
`instrumentEventConfiguration` has to be set on the portfolio at creation time. Passing
`recipe=RECIPE` into `recreate_portfolio()` back in section 3 took care of that. Skip it, and
`query_applicable_instrument_events` just comes back empty instead of raising an error.

Query a window that spans the swap's maturity and you'll get the `MaturityEvent` back, along with
the transaction it produces: both the asset leg and the financing leg close out on the swap's
`maturity_date`.

In [7]:
events_api = api(lusid.InstrumentEventsApi)

WINDOW_END = MATURITY + timedelta(days=5)

applicable = events_api.query_applicable_instrument_events(
    query_applicable_instrument_events_request=m.QueryApplicableInstrumentEventsRequest(
        window_start=ASOF.isoformat(),
        window_end=WINDOW_END.isoformat(),
        effective_at=WINDOW_END.isoformat(),
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=PORTFOLIO, portfolio_entity_type="SinglePortfolio")],
        forecasting_recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE))).values

display(pd.DataFrame([{
    "event type": ev.instrument_event_type,
    "eligible balance": ev.eligible_balance,
    "status": ev.instrument_event_status,
} for ev in applicable]))

,event type,eligible balance,status
0,MaturityEvent,"50,000.00",Active


---
# Summary

1. An asset TRS is a `TotalReturnSwap` whose asset leg carries the referenced obligation -- here
   that's a `Bond`, built from its own coupon and conventions.
2. The bond's `principal` has to match the funding leg's `notional`. Setting both to a unit value
   turns the swap into a per-unit instrument, so the position's quantity is what actually scales it.
3. It's reported at a quoted mark under `SimpleStatic`, so the legs describe what the instrument
   is rather than feeding into the price.
4. The swap's `MaturityEvent` works out of the box for forecasting, but only once the portfolio's
   `instrumentEventConfiguration` points at a recipe -- and that can only be set when the portfolio
   is created.

In [8]:
print(f"Scope      : {SCOPE}")
print(f"Portfolio  : {SCOPE}/{PORTFOLIO}")
print(f"Recipe     : {SCOPE}/{RECIPE}")
print(f"Instrument : {TRS_LUID}")

Scope      : AssetTrsDemo
Portfolio  : AssetTrsDemo/asset-trs-demo-book
Recipe     : AssetTrsDemo/asset-trs-demo-recipe
Instrument : LUID_00003DFP
